### Benchmark of loading efficiency solution to MODFLOW 6
Two aquifers. Left half is surface water, left half is land with three different top boundary conditions. Block response in river. 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import timflow.transient as tft

In [ ]:
# parameters of the problem
k = 20  # m/d
zconf = [0, -10, -12, -22]
zsemi = [1] + zconf
Ss = 1e-4  # 1/m
Sll = 1e-4  # 1/m
Sp = 0.1  # -
c = 500  # d
tsandhstar = [(0, 1), (1, 0)]
criv = 500  # d
leffaq = 0.7
leffll = 1

In [ ]:
# parameters needed in MODFLOW
nleaky = 10  # multiple leaky layers to simulate leakage out of clay layer
nlay = 2 + nleaky
delr = 10
delc = 1
nrow = 1
nleft = 200
nright = 200
ncol = nleft + nright
top = zconf[0]
botm = np.hstack((np.linspace(zconf[1], zconf[2], nleaky + 1), zconf[3]))
kmf = np.hstack((k, (botm[0] - botm[-2]) / c * np.ones(nleaky), k))
hinit = 0 * np.ones((nlay, 1, ncol))
# loading efficiency
hinit[0, 0, :nleft] = leffaq
hinit[-1, 0, :nleft] = leffaq
hinit[1:-1, 0, :nleft] = leffll
hleft = 1
hright = 0
cond = delr / c
ghb1 = []
for i in range(nleft):
    ghb1.append([(0, 0, i), hleft, cond])
ghb2 = []
for i in range(nleft):  # set back
    ghb2.append([(0, 0, i), 0, cond])
ghbland = []
for i in range(nleft, nleft + nright):  # set back
    ghbland.append([(0, 0, i), 0, cond])
S = Ss * np.ones((nlay, nrow, ncol))
S[1:nlay] = Sll
#
perlen = 1.0
nstep = 2400

### Function for MODFLOW model
Only run when MODFLOW 6 and Flopy are installed. In this notebook, the MODFLOW values are loaded from file. 

In [ ]:
# import flopy as fp

# def modflow(hinit, ghb_list, perlen, nstep, S):
#     # Define name and path
#     modelname = 'modflow' # model name to be used
#     modelws = './modflow' # model work space to be used

#     # Create simulation
#     sim = fp.mf6.MFSimulation(sim_name=modelname,
#                               version='mf6',
#                               exe_name='/Users/mark/bin/mf6', # change path
#                               sim_ws=modelws,
#                              )

#     # Time discretization
#     tdis = fp.mf6.ModflowTdis(simulation=sim,
#                               time_units='DAYS',
#                               nper=1,
#                               perioddata=[(perlen, nstep, 1)],
#                              )

#     # Iterative model solution
#     ims = fp.mf6.ModflowIms(simulation=sim,
#                             complexity='SIMPLE',
#                            )

#     # Groundwater flow model
#     gwf = fp.mf6.ModflowGwf(simulation=sim,
#                             modelname=modelname,
#                            )

#     # Spatial iscretization
#     dis = fp.mf6.ModflowGwfdis(model=gwf,
#                                length_units='METERS',
#                                xorigin=-2000,
#                                yorigin=0,
#                                nlay=nlay,
#                                nrow=nrow,
#                                ncol=ncol,
#                                delr=delr,
#                                delc=delc,
#                                top=top,
#                                botm=botm,
#                               )

#     # Aquifer properties
#     npf = fp.mf6.ModflowGwfnpf(model=gwf,
#                                icelltype=0,
#                                k=kmf,
#                               )

#     sto = fp.mf6.ModflowGwfsto(model=gwf,
#                                ss=S,
#                                transient=True,
#                               )

#     # Initial conditions
#     ic = fp.mf6.ModflowGwfic(model=gwf,
#                              strt=hinit,
#                             )

#     #for i in range(nright):
#     #    ghb_list.append([(0, 0, i + nleft), hright, cond])
#     ghb = fp.mf6.ModflowGwfghb(model=gwf,
#                                stress_period_data={0: ghb_list},
#                                pname='ghb'
#                               )

#     # Output control
#     oc = fp.mf6.ModflowGwfoc(model=gwf,
#                              budget_filerecord=f"{modelname}.cbc",
#                              head_filerecord=f"{modelname}.hds",
#                              saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
#                             )

#     # Write input files and solve
#     sim.write_simulation(silent=True)
#     success, _ = sim.run_simulation(silent=True)
#     if success == 1:
#         print('Model solved successfully')
#     else:
#         print('Solve failed')

#     # load head solution
#     hds = gwf.output.head() # get handle to binary head file
#     head = hds.get_alldata() # get the head data from the file
#     xc = gwf.modelgrid.xcellcenters[0]
#     return head[-1].squeeze(), xc

### Confined aquifer below the land

In [ ]:
# timflow model
ml = tft.ModelXsection(naq=2, tmin=0.0001, tmax=10)
riv = tft.XsectionMaq(
    model=ml,
    x1=-np.inf,
    x2=0,
    z=zsemi,
    kaq=k,
    Saq=Ss,
    Sll=[0, Sll],  # MODFLOW cannot do storage in ghb
    c=[criv, c],
    leffaq=leffaq,
    leffll=leffll,
    topboundary="semi",
    tsandhstar=tsandhstar,
    name="river",
)
land = tft.XsectionMaq(
    model=ml,
    x1=0,
    x2=np.inf,
    z=zconf,
    kaq=k,
    Saq=Ss,
    Sll=Sll,
    c=c,
    topboundary="conf",
    name="land",
)
ml.solve()

In [ ]:
x = np.linspace(-2000, 2000, 100)
t = 1
htim = ml.headalongline(x, 0, t)
# # compute modflow solution
# hmf, xmf = modflow(hinit, ghb1, perlen, nstep, S)
# np.savetxt('hmfconf1.dat', hmf)
# np.savetxt('xmf.dat', xmf)
# load modflow solution
hmf = np.loadtxt("./data/hmfconf1.dat")
xmf = np.loadtxt("./data/xmf.dat")

In [ ]:
ml.plots.xsection(xy=[(-1, 0), (1, 0)], params=True, fmt=".0f");

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(x, htim[0, 0], label="timflow top aquifer")
plt.plot(x, htim[1, 0], label="timflow bot aquifer")
plt.plot(xmf, hmf[0], "k--", label="modflow top aquifer")
plt.plot(xmf, hmf[-1], "k:", label="modflow bot aquifer")
plt.legend()
plt.title(f"aquifer below land is confined. head at t={t} d", fontsize=9)
plt.xlabel("x (m)")
plt.ylabel("head(m)")
plt.grid()

In [ ]:
t = 2
htim2 = ml.headalongline(x, 0, t)
#
# # compute modflow solution
# hinit2 = np.zeros((nlay, nrow, ncol))
# hinit2[:, 0] = hmf
# hinit2[0, 0, :nleft] -= leffaq
# hinit2[1:-1, 0, :nleft] -= leffll
# hinit2[-1, 0, :nleft] -= leffaq
# hmf2, xmf2 = modflow(hinit2, ghb2, perlen, nstep, S)
# np.savetxt('hmfconf2.dat', hmf2)
# load modflow solution
hmf2 = np.loadtxt("./data/hmfconf2.dat")

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(x, htim2[0, 0], label="timflow top aquifer")
plt.plot(x, htim2[1, 0], label="timflow bot aquifer")
plt.plot(xmf, hmf2[0], "k--", label="modflow top aquifer")
plt.plot(xmf, hmf2[-1], "k:", label="modflow bot aquifer")
plt.legend()
plt.title(f"aquifer below land is confined. head at t={t} d", fontsize=9)
plt.xlabel("x (m)")
plt.ylabel("head(m)")
plt.grid()

### Unconfined aquifer below the land

In [ ]:
# timflow model
ml = tft.ModelXsection(naq=2, tmin=0.0001, tmax=10)
riv = tft.XsectionMaq(
    model=ml,
    x1=-np.inf,
    x2=0,
    z=zsemi,
    kaq=k,
    Saq=Ss,
    Sll=[0, Sll],  # MODFLOW cannot do storage in ghb
    c=[criv, c],
    leffaq=leffaq,
    leffll=leffll,
    topboundary="semi",
    tsandhstar=tsandhstar,
    name="river",
)
land = tft.XsectionMaq(
    model=ml,
    x1=0,
    x2=np.inf,
    z=zconf,
    kaq=k,
    Saq=[Sp, Ss],
    Sll=Sll,
    c=c,
    topboundary="phreatic",
    name="land",
)
ml.solve()

In [ ]:
x = np.linspace(-2000, 2000, 100)
t = 1
htim = ml.headalongline(x, 0, t)
# # compute modflow
# Sphreatic = S.copy()
# Sphreatic[0, 0, nleft:] = Sp / (zconf[0] - zconf[1])
# hmf, xmf = modflow(hinit, ghb1, perlen, nstep, Sphreatic)
# np.savetxt('hmfphre1.dat', hmf)
# load modflow solution
hmf = np.loadtxt("./data/hmfphre1.dat")

In [ ]:
ml.plots.xsection(xy=[(-1, 0), (1, 0)], params=True, fmt=".1f");

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(x, htim[0, 0], label="timflow top aquifer")
plt.plot(x, htim[1, 0], label="timflow bot aquifer")
plt.plot(xmf, hmf[0], "k--", label="modflow top aquifer")
plt.plot(xmf, hmf[-1], "k:", label="modflow bot aquifer")
plt.legend()
plt.title(f"aquifer below land is unconfined. head at t={t} d", fontsize=9)
plt.xlabel("x (m)")
plt.ylabel("head(m)")
plt.grid()

In [ ]:
t = 2
htim2 = ml.headalongline(x, 0, t)
# # compute modflow
# hinit2 = np.zeros((nlay, nrow, ncol))
# hinit2[:, 0] = hmf
# hinit2[0, 0, :nleft] -= leffaq
# hinit2[1:-1, 0, :nleft] -= leffll
# hinit2[-1, 0, :nleft] -= leffaq
# hmf2, xmf2 = modflow(hinit2, ghb2, perlen, nstep, Sphreatic)
# np.savetxt('hmfphre2.dat', hmf2)
# load modflow solution
hmf2 = np.loadtxt("./data/hmfphre2.dat")

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(x, htim2[0, 0], label="timflow top aquifer")
plt.plot(x, htim2[1, 0], label="timflow bot aquifer")
plt.plot(xmf, hmf2[0], "k--", label="modflow top aquifer")
plt.plot(xmf, hmf2[-1], "k:", label="modflow bot aquifer")
plt.legend()
plt.title(f"aquifer below land is unconfined. head at t={t} d", fontsize=9)
plt.xlabel("x (m)")
plt.ylabel("head(m)")
plt.grid()

### Semi-confined aquifer below the land

In [ ]:
# timflow model
ml = tft.ModelXsection(naq=2, tmin=0.0001, tmax=10)
riv = tft.XsectionMaq(
    model=ml,
    x1=-np.inf,
    x2=0,
    z=zsemi,
    kaq=k,
    Saq=Ss,
    Sll=[0, Sll],  # MODFLOW cannot do storage in ghb
    c=[criv, c],
    leffaq=leffaq,
    leffll=leffll,
    topboundary="semi",
    tsandhstar=tsandhstar,
    name="river",
)
land = tft.XsectionMaq(
    model=ml,
    x1=0,
    x2=np.inf,
    z=zsemi,
    kaq=k,
    Saq=Ss,
    Sll=[0, Sll],
    c=c,
    topboundary="semi",
    name="land",
)
ml.solve()

In [ ]:
x = np.linspace(-2000, 2000, 100)
t = 1
htim = ml.headalongline(x, 0, t)
# # compute modflow
# hmf, xmf = modflow(hinit, ghb1 + ghbland, perlen, nstep, S)
# np.savetxt('hmfsemi1.dat', hmf)
# load modflow solution
hmf = np.loadtxt("./data/hmfsemi1.dat")

In [ ]:
ml.plots.xsection(xy=[(-1, 0), (1, 0)], params=True, fmt=".0f");

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(x, htim[0, 0], label="timflow top aquifer")
plt.plot(x, htim[1, 0], label="timflow bot aquifer")
plt.plot(xmf, hmf[0], "k--", label="modflow top aquifer")
plt.plot(xmf, hmf[-1], "k:", label="modflow bot aquifer")
plt.legend()
plt.title(f"head at t={t} d", fontsize=9)
plt.xlabel("x (m)")
plt.ylabel("head(m)")
plt.grid()

In [ ]:
t = 2
htim2 = ml.headalongline(x, 0, t)
# # compute modflow
# hinit2 = np.zeros((nlay, nrow, ncol))
# hinit2[:, 0] = hmf
# hinit2[0, 0, :nleft] -= leffaq
# hinit2[1:-1, 0, :nleft] -= leffll
# hinit2[-1, 0, :nleft] -= leffaq
# hmf2, xmf2 = modflow(hinit2, ghb2 + ghbland, perlen, nstep, S)
# np.savetxt('hmfsemi2.dat', hmf2)
# load modflow solution
hmf2 = np.loadtxt("./data/hmfsemi2.dat")

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(x, htim2[0, 0], label="timflow top aquifer")
plt.plot(x, htim2[1, 0], label="timflow bot aquifer")
plt.plot(xmf, hmf2[0], "k--", label="modflow top aquifer")
plt.plot(xmf, hmf2[-1], "k:", label="modflow bot aquifer")
plt.legend()
plt.title(f"head at t={t} d", fontsize=9)
plt.xlabel("x (m)")
plt.ylabel("head(m)")
plt.grid()